# Walk-Forward Out-of-Sample Evaluation and Long-Short Backtest

## Objective

Evaluate the frozen OLS, Ridge, Elastic Net, and XGBoost specifications
on the untouched 2022+ test period.

The evaluation uses an expanding annual walk-forward procedure:

- At the beginning of each year, fit each model using all observations
  whose targets would already have been known.
- Keep all previously selected hyperparameters fixed.
- Generate predictions for the coming year.
- Add the completed year to the training history before the next refit.

The final evaluation considers both:

1. Statistical predictive performance:
   - OOS R²
   - Pearson IC
   - Rank IC
   - year-by-year stability

2. Economic performance:
   - long top 20% / short bottom 20%
   - dollar-neutral, 100% gross exposure
   - turnover
   - 0, 5, and 10 bps transaction-cost assumptions
   - annualized return, volatility, Sharpe ratio, and maximum drawdown

No model specification or hyperparameter will be changed based on
test-period performance.

In [ ]:
import json

import numpy as np
import pandas as pd

import plotly.express as px


import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from metrics import (
    oos_r2,
    daily_ic,
    evaluate_predictions,
    performance_stats,
    max_drawdown,
    newey_west_mean_test,
    newey_west_paired_test,
)
from backtest import build_long_short_portfolio
from models import make_models


In [2]:
# Load the data
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

df = pd.read_csv(
    DATA_DIR / "alpha_dataset.csv",
    parse_dates=[
        "date",
        "next_market_date",
    ],
)

In [ ]:
# Load linear parameters
with open(DATA_DIR / "linear_model_params.json", "r") as f:
    linear_params = json.load(f)

linear_params

{'ridge': {'alpha': 300000.0},
 'elastic_net': {'alpha': 3e-05, 'l1_ratio': 0.9}}

In [ ]:
# Load XGBoost parameters
with open(DATA_DIR / "xgboost_params.json", "r") as f:
    xgb_params = json.load(f)

xgb_params

{'max_depth': 1,
 'learning_rate': 0.02,
 'n_estimators': 300,
 'subsample': 0.8,
 'colsample_bytree': 0.8,
 'min_child_weight': 10,
 'reg_lambda': 10.0,
 'reg_alpha': 0.0}

## Define features, target and evaluation functions

In [5]:
MODEL_FEATURES = [
    "ret_1d_z",
    "mom_5_z",
    "mom_20_z",
    "mom_60_z",
    "vol_20_z",
    "vol_60_z",
    "price_ma20_gap_z",
    "dollar_volume_z20_z",
    "beta_60_z",
]

TARGET = "target_cs_1d"

In [ ]:
required_cols = MODEL_FEATURES + [
    TARGET,
    "fwd_ret_1d",
    "next_market_date",
]

missing_cols = set(required_cols) - set(df.columns)

assert not missing_cols, f"Missing columns: {missing_cols}"

## Annual expanding-window procedure

* For the 2022 predictions: Fit all eligible data before the first 2022 trading day and predict all 2022 dates

* For the 2023 predictions: Fit all eligible data through 2022 and predict all 2023 dates

* For the 2024 predictions: Fit all eligible data through 2023 and predict all 2024 dates

and so on.

Hyperparameters remain fixed throughout.

This mimics a researcher who periodically retrains the model as new realized returns become available, without using future observations.

### Ridge regularization convention

The Ridge penalty is kept numerically fixed at the validation-selected value
throughout the walk-forward test.

Scikit-learn Ridge minimizes an objective of the form

$$
\|y-X\beta\|_2^2+\alpha\|\beta\|_2^2.
$$

Because the squared-error term grows with the number of training observations,
a fixed numerical value of alpha corresponds to weaker effective
regularization as the expanding training sample becomes larger.

A sample-size-normalized Ridge penalty could instead scale approximately as

$$
\alpha_t
=
\alpha_0
\frac{n_t}{n_0}.
$$

This was not introduced into the primary test after observing the final
results, because doing so would constitute a post-test methodological change.
It is therefore reported as a limitation / possible robustness extension.

## Generate walk-forward predictions

In [7]:
test_years = sorted(df.loc[df["split"] == "test", "date"].dt.year.unique())

test_years

[np.int32(2022),
 np.int32(2023),
 np.int32(2024),
 np.int32(2025),
 np.int32(2026)]

In [ ]:
prediction_parts = []
refit_log = []

for year in test_years:
    # -------------------------------
    # 1. Data we need to predict
    # -------------------------------

    year_data = df[(df["split"] == "test") & (df["date"].dt.year == year)].sort_values(["date", "ticker"]).copy()

    first_prediction_date = year_data["date"].min()

    # -------------------------------
    # 2. Training data available
    #    before this prediction year
    # -------------------------------

    train_data = (
        df[df["next_market_date"] < first_prediction_date]
        .dropna(subset=MODEL_FEATURES + [TARGET])
        .sort_values(["date", "ticker"])
        .copy()
    )

    X_train = train_data[MODEL_FEATURES]

    y_train = train_data[TARGET]

    X_year = year_data[MODEL_FEATURES]

    # -------------------------------
    # 3. Fresh frozen models
    # -------------------------------

    models = make_models(
        linear_params,
        xgb_params,
    )

    # Basic output for this year
    year_predictions = year_data[
        [
            "date",
            "next_market_date",
            "ticker",
            "fwd_ret_1d",
            TARGET,
        ]
    ].copy()

    # -------------------------------
    # 4. Fit and predict each model
    # -------------------------------

    for model_name, model in models.items():
        model.fit(X_train, y_train)

        year_predictions[f"{model_name}_pred"] = model.predict(X_year)

    prediction_parts.append(year_predictions)

    # Record exactly what data each
    # annual model was trained on
    refit_log.append(
        {
            "prediction_year": year,
            "first_prediction_date": first_prediction_date,
            "train_start": train_data["date"].min(),
            "train_end": train_data["date"].max(),
            "n_train_rows": len(train_data),
            "n_train_dates": train_data["date"].nunique(),
            "n_prediction_rows": len(year_data),
        }
    )

In [ ]:
# Combine
oos_predictions = pd.concat(prediction_parts, ignore_index=True).sort_values(["date", "ticker"]).reset_index(drop=True)

In [10]:
## Refit log
refit_log = pd.DataFrame(refit_log)

refit_log

,prediction_year,first_prediction_date,train_start,train_end,n_train_rows,n_train_dates,n_prediction_rows
0,2022,2022-01-03,2010-03-31,2021-12-30,88800,2960,7530
1,2023,2023-01-03,2010-03-31,2022-12-29,96330,3211,7500
2,2024,2024-01-02,2010-03-31,2023-12-28,103830,3461,7560
3,2025,2025-01-02,2010-03-31,2024-12-30,111390,3713,7500
4,2026,2026-01-02,2010-03-31,2025-12-30,118890,3963,4890


## Inspect predictions

In [11]:
oos_predictions.head()

,date,next_market_date,ticker,fwd_ret_1d,target_cs_1d,ols_pred,ridge_pred,elastic_net_pred,xgboost_pred
0,2022-01-03,2022-01-04,AAPL,-0.012692,-0.021930,0.000098,0.000006,0.000047,0.000159
1,2022-01-03,2022-01-04,AMGN,0.005073,-0.004165,0.000204,0.000006,0.000045,0.000071
2,2022-01-03,2022-01-04,BA,0.027759,0.018521,0.000023,0.000013,0.000008,-0.000011
3,2022-01-03,2022-01-04,BAC,0.039194,0.029956,-0.000304,-0.000056,-0.000141,-0.000099
4,2022-01-03,2022-01-04,C,0.007766,-0.001473,-0.000469,-0.000077,-0.000215,-0.000150


In [12]:
oos_predictions[
    [
        "ols_pred",
        "ridge_pred",
        "elastic_net_pred",
        "xgboost_pred",
    ]
].describe()

,ols_pred,ridge_pred,elastic_net_pred,xgboost_pred
count,3.498000e+04,3.498000e+04,3.498000e+04,34980.000000
mean,5.552755e-19,5.552600e-19,5.549206e-19,-0.000005
std,1.473167e-04,3.411548e-05,6.315591e-05,0.000271
min,-6.881489e-04,-1.696784e-04,-3.904571e-04,-0.003919
25%,-9.359038e-05,-2.150864e-05,-3.683571e-05,-0.000025
50%,-8.505650e-06,-1.592701e-06,4.574665e-07,0.000021
75%,8.218618e-05,1.916784e-05,3.655173e-05,0.000048
max,9.380461e-04,1.823968e-04,3.005319e-04,0.003485


In [ ]:
print(
    "Dates:",
    oos_predictions["date"].min(),
    "to",
    oos_predictions["date"].max(),
)

print("Number of dates:", oos_predictions["date"].nunique())

Dates: 2022-01-03 00:00:00 to 2026-08-26 00:00:00
Number of dates: 1166


## Overall test-period prediction performance

In [14]:
PREDICTION_COLS = {
    "OLS": "ols_pred",
    "Ridge": "ridge_pred",
    "Elastic Net": "elastic_net_pred",
    "XGBoost": "xgboost_pred",
}

In [ ]:
test_results = []

for model_name, pred_col in PREDICTION_COLS.items():
    metrics = evaluate_predictions(oos_predictions, oos_predictions[pred_col].values, model_name)

    test_results.append(metrics)

In [ ]:
test_results = pd.DataFrame(test_results)

test_results.sort_values("mean_rank_ic", ascending=False)

,model,mse,oos_r2,mean_pearson_ic,mean_rank_ic,rank_ic_std,pct_rank_ic_positive
3,XGBoost,0.000261,-0.000524,-0.007767,-0.001214,0.240532,0.504288
1,Ridge,0.000261,-0.000022,-0.003708,-0.004332,0.301862,0.506003
2,Elastic Net,0.000261,-0.000080,-0.005779,-0.005730,0.274356,0.486278
0,OLS,0.000261,-0.000223,-0.011265,-0.013070,0.264869,0.486278


## Statistical inference for final test Rank IC

Unlike the validation period, the 2022+ test period was not used for model
or hyperparameter selection.

For each frozen model, we test whether mean daily Rank IC differs from zero
using Newey-West / HAC standard errors. We also test selected paired
differences in daily IC between models.

In [ ]:
test_ic_series_dict = {}

for model_name, pred_col in PREDICTION_COLS.items():
    test_ic_series_dict[model_name] = daily_ic(
        oos_predictions,
        prediction_col=pred_col,
        target_col=TARGET,
        method="spearman",
    )

In [ ]:
test_ic_inference = []

for model_name, ic in test_ic_series_dict.items():
    result = newey_west_mean_test(ic)

    result["model"] = model_name

    test_ic_inference.append(result)


test_ic_inference = pd.DataFrame(test_ic_inference)[
    [
        "model",
        "n",
        "mean",
        "nw_se",
        "nw_t",
        "nw_pvalue",
        "maxlags",
    ]
]

test_ic_inference

,model,n,mean,nw_se,nw_t,nw_pvalue,maxlags
0,OLS,1166,-0.013070,0.007920,-1.650100,0.098923,6
1,Ridge,1166,-0.004332,0.008774,-0.493713,0.621509,6
2,Elastic Net,1166,-0.005730,0.008061,-0.710818,0.477197,6
3,XGBoost,1166,-0.001214,0.007667,-0.158395,0.874145,6


In [ ]:
test_pairs = [
    (
        "Ridge",
        "OLS",
    ),
    (
        "Elastic Net",
        "OLS",
    ),
    (
        "XGBoost",
        "Ridge",
    ),
]

test_pair_results = []

for model_a, model_b in test_pairs:
    result = newey_west_paired_test(
        test_ic_series_dict[model_a],
        test_ic_series_dict[model_b],
    )

    result["comparison"] = f"{model_a} - {model_b}"

    test_pair_results.append(result)


test_pair_results = pd.DataFrame(test_pair_results)[
    [
        "comparison",
        "n",
        "mean_a",
        "mean_b",
        "mean_difference",
        "nw_se",
        "nw_t",
        "nw_pvalue",
        "maxlags",
    ]
]

test_pair_results

,comparison,n,mean_a,mean_b,mean_difference,nw_se,nw_t,nw_pvalue,maxlags
0,Ridge - OLS,1166,-0.004332,-0.013070,0.008738,0.005525,1.581566,0.113749,6
1,Elastic Net - OLS,1166,-0.005730,-0.013070,0.007340,0.005159,1.422669,0.154832,6
2,XGBoost - Ridge,1166,-0.001214,-0.004332,0.003117,0.006886,0.452695,0.650768,6


### Statistical interpretation of the final test

None of the four frozen models produces statistically detectable mean Rank IC
during the 2022–2026 test period.

Newey-West results are:

- OLS: Rank IC = -0.0131, p = 0.099
- Ridge: Rank IC = -0.0043, p = 0.622
- Elastic Net: Rank IC = -0.0057, p = 0.477
- XGBoost: Rank IC = -0.0012, p = 0.874

Pairwise differences between the models are also statistically insignificant.

Therefore, the final test does not support the existence of persistent
cross-sectional predictive ability in any of the frozen specifications.

This is more precise than saying that a previously established signal
"decayed": the validation estimates themselves were noisy and not statistically
distinguishable from zero. The correct interpretation is that positive
validation point estimates did not translate into detectable future
predictability.

## Performance by year

In [ ]:
yearly_prediction_results = []

for year, d in oos_predictions.groupby(oos_predictions["date"].dt.year):
    for model_name, pred_col in PREDICTION_COLS.items():
        metrics = evaluate_predictions(
            d,
            d[pred_col].values,
            model_name,
        )

        metrics["year"] = year

        yearly_prediction_results.append(metrics)

In [ ]:
yearly_prediction_results = pd.DataFrame(yearly_prediction_results)

yearly_prediction_results[
    [
        "year",
        "model",
        "oos_r2",
        "mean_rank_ic",
        "pct_rank_ic_positive",
    ]
]

,year,model,oos_r2,mean_rank_ic,pct_rank_ic_positive
0,2022,OLS,-0.000375,-0.018656,0.501992
1,2022,Ridge,-0.000154,-0.032684,0.486056
2,2022,Elastic Net,-0.000320,-0.023645,0.490040
3,2022,XGBoost,-0.000740,-0.039791,0.462151
4,2023,OLS,-0.000480,-0.027807,0.448000
5,2023,Ridge,0.000021,-0.015550,0.480000
6,2023,Elastic Net,0.000023,-0.009547,0.472000
7,2023,XGBoost,0.000178,-0.014541,0.440000
8,2024,OLS,-0.000320,-0.013329,0.476190
9,2024,Ridge,-0.000058,-0.000597,0.492063


In [ ]:
yearly_rank_ic = yearly_prediction_results.pivot(
    index="year",
    columns="model",
    values="mean_rank_ic",
)

yearly_rank_ic

model,Elastic Net,OLS,Ridge,XGBoost
year,,,,
2022,-0.023645,-0.018656,-0.032684,-0.039791
2023,-0.009547,-0.027807,-0.015550,-0.014541
2024,-0.001903,-0.013329,-0.000597,0.014271
2025,0.006870,-0.001486,0.018596,0.036534
2026,0.002469,0.000771,0.015593,-0.003208


In [ ]:
# Plot annual Rank IC
rank_ic_plot = yearly_prediction_results[
    [
        "year",
        "model",
        "mean_rank_ic",
    ]
]

fig = px.line(
    rank_ic_plot,
    x="year",
    y="mean_rank_ic",
    color="model",
    markers=True,
    title="Out-of-Sample Rank IC by Year",
)

fig.add_hline(
    y=0,
    line_dash="dash",
)

fig.show()

---

## Convert scores into portfolio weights

Our portfolio rule is: $$ \text{Long} : \text{Top 20\%} $$
 $$ \text{Short} : \text{Bottom 20\% } $$

So for 30 stocks: $$ 30 \times 0.20 = 6 $$
So each day we'll typically hold:
* 6 long stocks
* 6 short stocks

We will use $ 100\% $ gross exposure: $$ \sum_{i} |w_i| = 1 $$
That means:  $$ \sum_{\text{long}} w_i = +0.5 $$
 and $$ \sum_{\text{short}} w_i = -0.5 $$
Thus the portfolio is: $$ \text{Dollar Neutral} $$
with $$ \sum_{i} w_i = 0 $$

## Build all four portfolios

In [ ]:
portfolios = {}

for model_name, pred_col in PREDICTION_COLS.items():
    portfolios[model_name] = build_long_short_portfolio(
        oos_predictions,
        prediction_col=pred_col,
        long_fraction=0.20,
    )

In [ ]:
# Inspect XGBoost
portfolios["XGBoost"].head()

,signal_date,return_date,gross_return,long_return,short_return,turnover,n_long,n_short
0,2022-01-03,2022-01-04,-0.019689,-0.008681,-0.011009,1.000000,6,6
1,2022-01-04,2022-01-05,0.000562,-0.004133,0.004695,0.666667,6,6
2,2022-01-05,2022-01-06,-0.002994,-0.000922,-0.002072,0.666667,6,6
3,2022-01-06,2022-01-07,0.008145,0.002145,0.006000,0.666667,6,6
4,2022-01-07,2022-01-10,-0.000670,0.003310,-0.003980,0.833333,6,6


In [26]:
# Check exposure indirectly
portfolios["XGBoost"][
    [
        "gross_return",
        "turnover",
        "n_long",
        "n_short",
    ]
].describe()

,gross_return,turnover,n_long,n_short
count,1166.000000,1166.000000,1166.0,1166.0
mean,0.000023,0.842767,6.0,6.0
std,0.006184,0.297192,0.0,0.0
min,-0.032363,0.166667,6.0,6.0
25%,-0.003352,0.666667,6.0,6.0
50%,0.000064,0.833333,6.0,6.0
75%,0.003617,1.000000,6.0,6.0
max,0.026599,1.833333,6.0,6.0


## Transaction costs

We'll test: $$ 0, 5, 10 \text{ bps} $$ per dollar traded

In [ ]:
COST_BPS = [0, 5, 10]

for model_name, p in portfolios.items():
    for cost_bps in COST_BPS:
        cost_rate = cost_bps / 10_000

        p[f"net_return_{cost_bps}bps"] = p["gross_return"] - cost_rate * p["turnover"]

## Backtest implementation audit

Before interpreting performance, the main backtest mechanics are independently
reconstructed and checked:

- next-day return alignment;
- zero net exposure;
- 100% gross exposure;
- stock-level portfolio-return aggregation;
- target-weight turnover;
- transaction-cost arithmetic.

These are implementation checks only and do not alter the strategy.

In [28]:
# ============================================================
# 1. Independently reconstruct forward returns from prices
# ============================================================

raw_prices = pd.read_csv(
    DATA_DIR / "raw_prices.csv",
    parse_dates=["date"],
)

price_lookup = raw_prices[
    [
        "date",
        "ticker",
        "adj_close",
    ]
].copy()


return_audit = oos_predictions[
    [
        "date",
        "next_market_date",
        "ticker",
        "fwd_ret_1d",
    ]
].copy()


current_prices = price_lookup.rename(columns={"adj_close": "adj_close_t"})

return_audit = return_audit.merge(
    current_prices,
    on=[
        "date",
        "ticker",
    ],
    how="left",
    validate="many_to_one",
)


next_prices = price_lookup.rename(
    columns={
        "date": "next_market_date",
        "adj_close": "adj_close_next",
    }
)

return_audit = return_audit.merge(
    next_prices,
    on=[
        "next_market_date",
        "ticker",
    ],
    how="left",
    validate="many_to_one",
)


return_audit["reconstructed_fwd_ret"] = return_audit["adj_close_next"] / return_audit["adj_close_t"] - 1


assert return_audit["adj_close_t"].notna().all()

assert return_audit["adj_close_next"].notna().all()

assert np.allclose(
    return_audit["fwd_ret_1d"],
    return_audit["reconstructed_fwd_ret"],
    atol=1e-12,
)


# ============================================================
# 2. Independently reconstruct portfolio mechanics
# ============================================================


def reconstruct_portfolio(
    predictions,
    prediction_col,
    long_fraction=0.20,
):

    rows = []

    previous_weights = {}

    for date, d in predictions.sort_values(
        [
            "date",
            "ticker",
        ]
    ).groupby("date"):
        d = d.dropna(
            subset=[
                prediction_col,
                "fwd_ret_1d",
            ]
        ).copy()

        d = d.sort_values(
            [
                prediction_col,
                "ticker",
            ]
        )

        n_stocks = len(d)

        n_side = max(
            1,
            int(np.floor(n_stocks * long_fraction)),
        )

        current_weights = {ticker: 0.0 for ticker in d["ticker"]}

        for ticker in d.head(n_side)["ticker"]:
            current_weights[ticker] = -0.5 / n_side

        for ticker in d.tail(n_side)["ticker"]:
            current_weights[ticker] = 0.5 / n_side

        weights = d["ticker"].map(current_weights)

        # Exposure checks
        net_exposure = weights.sum()

        gross_exposure = weights.abs().sum()

        assert np.isclose(
            net_exposure,
            0.0,
            atol=1e-12,
        )

        assert np.isclose(
            gross_exposure,
            1.0,
            atol=1e-12,
        )

        # Return reconstruction
        gross_return = (weights * d["fwd_ret_1d"]).sum()

        # Turnover reconstruction
        all_tickers = set(previous_weights) | set(current_weights)

        turnover = sum(
            abs(
                current_weights.get(
                    ticker,
                    0.0,
                )
                - previous_weights.get(
                    ticker,
                    0.0,
                )
            )
            for ticker in all_tickers
        )

        rows.append(
            {
                "signal_date": date,
                "gross_return_reconstructed": gross_return,
                "turnover_reconstructed": turnover,
                "net_exposure": net_exposure,
                "gross_exposure": gross_exposure,
            }
        )

        previous_weights = current_weights.copy()

    return pd.DataFrame(rows)


# ============================================================
# 3. Compare reconstructed portfolio to stored portfolio
# ============================================================

for model_name, pred_col in PREDICTION_COLS.items():
    reconstructed = reconstruct_portfolio(
        oos_predictions,
        prediction_col=pred_col,
        long_fraction=0.20,
    )

    stored = portfolios[model_name].copy()

    check = stored.merge(
        reconstructed,
        on="signal_date",
        how="inner",
        validate="one_to_one",
    )

    assert len(check) == len(stored) == len(reconstructed)

    assert np.allclose(
        check["gross_return"],
        check["gross_return_reconstructed"],
        atol=1e-12,
    )

    assert np.allclose(
        check["turnover"],
        check["turnover_reconstructed"],
        atol=1e-12,
    )

    assert np.allclose(
        check["net_exposure"],
        0.0,
        atol=1e-12,
    )

    assert np.allclose(
        check["gross_exposure"],
        1.0,
        atol=1e-12,
    )

    # Transaction-cost check
    for cost_bps in COST_BPS:
        expected_net = check["gross_return"] - check["turnover"] * cost_bps / 10_000

        assert np.allclose(
            check[f"net_return_{cost_bps}bps"],
            expected_net,
            atol=1e-12,
        )


print("PASS: Forward returns, exposures, portfolio returns, turnover, and transaction-cost calculations all verified.")

PASS: Forward returns, exposures, portfolio returns, turnover, and transaction-cost calculations all verified.


### Audit conclusion

All implementation checks passed to floating-point precision.

The stored forward returns, portfolio exposures, stock-level return aggregation,
target-weight turnover, and transaction-cost calculations were independently
reconstructed and matched the backtest outputs.

This reduces the likelihood that the reported performance is driven by an
obvious alignment or portfolio-accounting error. It does not remove the
execution and market-microstructure limitations discussed below.

## Final portfolio-results table

In [ ]:
portfolio_results = []

for model_name, p in portfolios.items():
    for cost_bps in COST_BPS:
        return_col = f"net_return_{cost_bps}bps"

        stats = performance_stats(
            p[return_col],
            turnover=p["turnover"],
        )

        stats["model"] = model_name

        stats["cost_bps"] = cost_bps

        portfolio_results.append(stats)

In [ ]:
portfolio_results = pd.DataFrame(portfolio_results)

portfolio_results = portfolio_results[
    [
        "model",
        "cost_bps",
        "annual_return",
        "annual_vol",
        "sharpe",
        "max_drawdown",
        "avg_daily_turnover",
        "annualized_turnover",
    ]
]

portfolio_results

,model,cost_bps,annual_return,annual_vol,sharpe,max_drawdown,avg_daily_turnover,annualized_turnover
0,OLS,0,-0.053687,0.104565,-0.475448,-0.235434,0.768010,193.538593
1,OLS,5,-0.141018,0.104701,-1.399076,-0.510135,0.768010,193.538593
2,OLS,10,-0.220322,0.104876,-2.319451,-0.686198,0.768010,193.538593
3,Ridge,0,-0.008979,0.122385,-0.012572,-0.153321,0.758148,191.053173
4,Ridge,5,-0.099291,0.122407,-0.792971,-0.420194,0.758148,191.053173
5,Ridge,10,-0.181406,0.122467,-1.572596,-0.615266,0.758148,191.053173
6,Elastic Net,0,-0.045820,0.111365,-0.365435,-0.213257,0.907233,228.622642
7,Elastic Net,5,-0.148930,0.111338,-1.392226,-0.531409,0.907233,228.622642
8,Elastic Net,10,-0.240941,0.111361,-2.418426,-0.723651,0.907233,228.622642
9,XGBoost,0,0.000882,0.098172,0.058095,-0.223276,0.842767,212.377358


## Cost sensitivity table

It answers the question: Does the strategy survive plausible trading costs?

In [ ]:
sharpe_by_cost = portfolio_results.pivot(
    index="model",
    columns="cost_bps",
    values="sharpe",
)

sharpe_by_cost.columns = [f"{c} bps" for c in sharpe_by_cost.columns]

sharpe_by_cost

,0 bps,5 bps,10 bps
model,,,
Elastic Net,-0.365435,-1.392226,-2.418426
OLS,-0.475448,-1.399076,-2.319451
Ridge,-0.012572,-0.792971,-1.572596
XGBoost,0.058095,-1.021620,-2.096039


## Cumulative returns at 5 bps

In [ ]:
portfolio_plot_parts = []

for model_name, p in portfolios.items():
    temp = p[
        [
            "return_date",
            "net_return_5bps",
        ]
    ].copy()

    temp["wealth"] = (1 + temp["net_return_5bps"]).cumprod()

    temp["model"] = model_name

    portfolio_plot_parts.append(temp)

In [ ]:
# Combine
portfolio_plot = pd.concat(portfolio_plot_parts, ignore_index=True)

In [ ]:
# Plot
fig = px.line(
    portfolio_plot,
    x="return_date",
    y="wealth",
    color="model",
    title=("Walk-Forward Long-Short Portfolio Growth — 5 bps Transaction Costs"),
)

fig.show()

## Portfolio performance by year

In [ ]:
yearly_portfolio_results = []

for model_name, p in portfolios.items():
    p = p.copy()

    p["year"] = pd.to_datetime(p["return_date"]).dt.year

    for year, d in p.groupby("year"):
        stats = performance_stats(
            d["net_return_5bps"],
            turnover=d["turnover"],
        )

        stats["model"] = model_name

        stats["year"] = year

        yearly_portfolio_results.append(stats)

In [ ]:
# Create table
yearly_portfolio_results = pd.DataFrame(yearly_portfolio_results)

yearly_portfolio_results[
    [
        "year",
        "model",
        "annual_return",
        "annual_vol",
        "sharpe",
        "max_drawdown",
    ]
]

,year,model,annual_return,annual_vol,sharpe,max_drawdown
0,2022,OLS,-0.197513,0.109433,-1.955290,-0.186706
1,2023,OLS,-0.077690,0.070993,-1.103658,-0.126553
2,2024,OLS,-0.152067,0.082840,-1.949252,-0.177832
3,2025,OLS,-0.095582,0.108139,-0.875259,-0.110743
4,2026,OLS,-0.193795,0.154398,-1.317665,-0.170125
5,2022,Ridge,-0.209157,0.121504,-1.869874,-0.203492
6,2023,Ridge,-0.071546,0.075937,-0.939638,-0.144263
7,2024,Ridge,-0.099678,0.100994,-0.989178,-0.126429
8,2025,Ridge,-0.059234,0.121773,-0.440886,-0.091689
9,2026,Ridge,-0.018029,0.192910,0.001788,-0.166664


In [ ]:
# Pivot Sharpe
yearly_sharpe = yearly_portfolio_results.pivot(
    index="year",
    columns="model",
    values="sharpe",
)

yearly_sharpe

model,Elastic Net,OLS,Ridge,XGBoost
year,,,,
2022,-2.508492,-1.955290,-1.869874,-3.306383
2023,-1.215936,-1.103658,-0.939638,-0.695216
2024,-1.359512,-1.949252,-0.989178,-0.298028
2025,-0.898538,-0.875259,-0.440886,0.727678
2026,-1.198768,-1.317665,0.001788,-1.888912


## Long vs short contributions

In [ ]:
leg_results = []

for model_name, p in portfolios.items():
    leg_results.append(
        {
            "model": model_name,
            "annualized_long_contribution": p["long_return"].mean() * 252,
            "annualized_short_contribution": p["short_return"].mean() * 252,
            "annualized_gross_return": p["gross_return"].mean() * 252,
        }
    )

In [ ]:
# Table
leg_results = pd.DataFrame(leg_results)

leg_results

,model,annualized_long_contribution,annualized_short_contribution,annualized_gross_return
0,OLS,0.044801,-0.094516,-0.049715
1,Ridge,0.050546,-0.052084,-0.001539
2,Elastic Net,0.016574,-0.057271,-0.040697
3,XGBoost,0.066553,-0.060850,0.005703


## Diagnostic 1 - Validation vs test Rank IC

This compares the observed validation and test-period mean Rank IC.

A positive validation mean that becomes approximately zero or negative in the
future sample indicates that the validation result did not reproduce. Because
validation estimates are noisy and were used for model selection, this should
not be interpreted as proof that a stable alpha existed and subsequently
"decayed."

In [ ]:
# Load the validation predictions saved form notebook 04

validation_predictions = pd.read_csv(DATA_DIR / "model_validation_predictions.csv", parse_dates=["date"])

In [41]:
# Use the same prediction-column mapping

PREDICTION_COLS = {
    "OLS": "ols_pred",
    "Ridge": "ridge_pred",
    "Elastic Net": "elastic_net_pred",
    "XGBoost": "xgboost_pred",
}

In [ ]:
# Compute validation and test Rank IC

ic_period_comparison = []

for model_name, pred_col in PREDICTION_COLS.items():
    # Validation-period IC
    val_ic = daily_ic(
        validation_predictions,
        prediction_col=pred_col,
        target_col=TARGET,
        method="spearman",
    )

    # Test-period IC
    test_ic = daily_ic(
        oos_predictions,
        prediction_col=pred_col,
        target_col=TARGET,
        method="spearman",
    )

    ic_period_comparison.append(
        {
            "model": model_name,
            "validation_rank_ic": val_ic.mean(),
            "test_rank_ic": test_ic.mean(),
        }
    )


ic_period_comparison = pd.DataFrame(ic_period_comparison)

ic_period_comparison

,model,validation_rank_ic,test_rank_ic
0,OLS,0.002137,-0.013070
1,Ridge,0.009320,-0.004332
2,Elastic Net,0.008628,-0.005730
3,XGBoost,0.015520,-0.001214


In [ ]:
# Plot
ic_long = ic_period_comparison.melt(
    id_vars="model",
    var_name="period",
    value_name="rank_ic",
)

fig = px.bar(
    ic_long,
    x="model",
    y="rank_ic",
    color="period",
    barmode="group",
    title="Validation vs Test Rank IC",
)

fig.add_hline(y=0, line_dash="dash")

fig.show()

## Diagnostic 2 - Rolling 60-day test Rank IC

This shows how predictability changes through time.

In [ ]:
test_ic_frames = []

for model_name, pred_col in PREDICTION_COLS.items():
    ic = daily_ic(
        oos_predictions,
        prediction_col=pred_col,
        target_col=TARGET,
        method="spearman",
    )

    temp = pd.DataFrame(
        {
            "date": ic.index,
            "rank_ic": ic.values,
            "model": model_name,
        }
    )

    test_ic_frames.append(temp)


test_ic_series = pd.concat(test_ic_frames, ignore_index=True)

In [ ]:
# Sort

test_ic_series = test_ic_series.sort_values(["model", "date"])

In [ ]:
# Calculate the rolling mean

test_ic_series["rank_ic_60d"] = test_ic_series.groupby("model")["rank_ic"].transform(
    lambda s: s.rolling(window=60, min_periods=20).mean()
)

In [ ]:
# Plot

fig = px.line(
    test_ic_series,
    x="date",
    y="rank_ic_60d",
    color="model",
    title="60-Day Rolling Rank IC — Test Period",
)

fig.add_hline(y=0, line_dash="dash")

fig.show()

## Diagnostic 3 - Turnover

In [ ]:
# Create a combined dataframe

turnover_data = []

for model_name, p in portfolios.items():
    temp = p[
        [
            "signal_date",
            "turnover",
        ]
    ].copy()

    temp["model"] = model_name

    turnover_data.append(temp)


turnover_data = pd.concat(turnover_data, ignore_index=True)

In [ ]:
# Summary table

turnover_summary = turnover_data.groupby("model")["turnover"].agg(
    mean="mean",
    median="median",
    std="std",
    min="min",
    max="max",
)

turnover_summary

,mean,median,std,min,max
model,,,,,
Elastic Net,0.907233,0.833333,0.295966,0.000000,2.000000
OLS,0.768010,0.833333,0.254754,0.166667,1.833333
Ridge,0.758148,0.666667,0.271817,0.166667,1.666667
XGBoost,0.842767,0.833333,0.297192,0.166667,1.833333


In [ ]:
# Add annualized turnover

turnover_summary["annualized_turnover"] = turnover_summary["mean"] * 252

turnover_summary

,mean,median,std,min,max,annualized_turnover
model,,,,,,
Elastic Net,0.907233,0.833333,0.295966,0.000000,2.000000,228.622642
OLS,0.768010,0.833333,0.254754,0.166667,1.833333,193.538593
Ridge,0.758148,0.666667,0.271817,0.166667,1.666667,191.053173
XGBoost,0.842767,0.833333,0.297192,0.166667,1.833333,212.377358


In [51]:
# Plot the distribution

fig = px.box(
    turnover_data,
    x="model",
    y="turnover",
    title="Daily Portfolio Turnover — Test Period",
)

fig.show()

## Save final predictions

In [ ]:
oos_predictions.to_csv(
    DATA_DIR / "oos_predictions.csv",
    index=False,
)

## Save portfolio daily returns

In [ ]:
portfolio_output = []

for model_name, p in portfolios.items():
    temp = p.copy()

    temp["model"] = model_name

    portfolio_output.append(temp)

In [ ]:
portfolio_output = pd.concat(portfolio_output, ignore_index=True)

portfolio_output.to_csv(
    DATA_DIR / "portfolio_returns.csv",
    index=False,
)

In [ ]:
# Save summary
portfolio_results.to_csv(
    DATA_DIR / "portfolio_summary.csv",
    index=False,
)

## Execution assumption

The primary backtest uses features constructed with end-of-day information
through date \(t\), including the date-\(t\) closing price, while evaluating
the adjusted-close return from \(t\) to the next trading day.

This effectively assumes that the strategy can trade at or very near the same
close used to finish constructing the signal. Because the official closing
price is not known before that execution price is available, this is an
optimistic execution assumption rather than a fully implementable trading
simulation.

This issue is especially relevant for short-horizon reversal signals, where
performance may depend strongly on execution timing.

The primary results should therefore be interpreted as a research backtest,
not as a production execution simulation.

A post-hoc robustness check below delays the signal by one trading day. This
check was added after inspecting the final test results and is therefore
reported only as a robustness diagnostic, not as a new primary specification.

## Post-hoc one-day signal-delay robustness check

In [ ]:
lagged_predictions = oos_predictions.sort_values(
    [
        "ticker",
        "date",
    ]
).copy()

for model_name, pred_col in PREDICTION_COLS.items():
    lagged_predictions[f"{pred_col}_lag1"] = lagged_predictions.groupby("ticker")[pred_col].shift(1)


lagged_predictions = lagged_predictions.sort_values(
    [
        "date",
        "ticker",
    ]
).reset_index(drop=True)

In [ ]:
lagged_prediction_results = []

for model_name, pred_col in PREDICTION_COLS.items():
    lag_col = f"{pred_col}_lag1"

    temp = lagged_predictions.dropna(
        subset=[
            lag_col,
            TARGET,
        ]
    ).copy()

    metrics = evaluate_predictions(
        temp,
        temp[lag_col].to_numpy(),
        f"{model_name} - lagged 1 day",
    )

    ic = daily_ic(
        temp,
        prediction_col=lag_col,
        target_col=TARGET,
        method="spearman",
    )

    inference = newey_west_mean_test(ic)

    metrics["nw_rank_ic_t"] = inference["nw_t"]

    metrics["nw_rank_ic_pvalue"] = inference["nw_pvalue"]

    lagged_prediction_results.append(metrics)


lagged_prediction_results = pd.DataFrame(lagged_prediction_results)

lagged_prediction_results

,model,mse,oos_r2,mean_pearson_ic,mean_rank_ic,rank_ic_std,pct_rank_ic_positive,nw_rank_ic_t,nw_rank_ic_pvalue
0,OLS - lagged 1 day,0.000261,-0.000110,-0.006025,-0.010221,0.267908,0.491845,-1.276235,0.201872
1,Ridge - lagged 1 day,0.000261,0.000006,-0.000546,-0.003414,0.305944,0.503004,-0.387128,0.698662
2,Elastic Net - lagged 1 day,0.000261,-0.000017,-0.001966,-0.003092,0.275716,0.500429,-0.376851,0.706285
3,XGBoost - lagged 1 day,0.000261,-0.000644,0.003064,0.007119,0.239101,0.516738,0.993007,0.320707


### One-day signal-delay robustness

As a post-hoc execution robustness check, all model scores were delayed by
one trading day before being evaluated against future returns.

The statistical conclusion remains unchanged.

XGBoost's mean Rank IC increased from approximately -0.0012 in the primary
test to approximately +0.0071 under the one-day-delayed signal, but the
Newey-West t-statistic was only about 0.99 with p ≈ 0.32.

Thus, delaying the signal does not provide statistically detectable evidence
of alpha. The robustness check also suggests that the project's main negative
conclusion is not being driven solely by the optimistic same-close execution
assumption.

Because this diagnostic was introduced after inspecting the final test
results, it is not treated as a new model specification or trading strategy.